In [1]:
#here we just need to run the chimera on our pdbs to save the aligned file in a folder and then calculate the rmsd and the distance between M and D

In [ ]:
import os
import csv
from chimerax.core.commands import run

def setup_directories():
    base_dir = os.path.dirname(os.path.abspath(__file__))
    malaria_dir = os.path.join(base_dir, "malariapdb_renamed")
    target_dir = os.path.join(base_dir, "targetpdb_renamed")
    log_dir = os.path.join(base_dir, "logs")

    os.makedirs(log_dir, exist_ok=True)

    return base_dir, malaria_dir, target_dir, log_dir

def align_and_save(session, malaria_path, target_path, count, log_dir):
    run(session, "log clear")
    run(session, f"open {target_path}")
    run(session, f"open {malaria_path}")
    run(session, "matchmaker #2 to #1")

    # Define save path relative to the target file
    target_dir = os.path.dirname(target_path)
    aligned_subdir = os.path.join(target_dir, "aligned")
    os.makedirs(aligned_subdir, exist_ok=True)

    target_name = os.path.basename(target_path).replace('.pdb', '')
    malaria_name = os.path.basename(malaria_path).replace('.pdb', '')
    aligned_filename = f"aligned_{count}_{target_name}_vs_{malaria_name}.pdb"
    aligned_path = os.path.join(aligned_subdir, aligned_filename)

    # Save log in central log directory
    log_path = os.path.join(log_dir, f"output{count}.txt")

    run(session, f"save {aligned_path}")
    run(session, f"log save {log_path}")
    run(session, "close all")

def main(session):
    base_dir, malaria_dir, target_dir, log_dir = setup_directories()
    csv_path = os.path.join(base_dir, "Ultimate_Dataset.csv")

    with open(csv_path, newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        for count, row in enumerate(reader, start=1):
            malaria_file = os.path.join(malaria_dir, row["malaria_renamed"])
            target_file = os.path.join(target_dir, row["target_renamed"])

            if not os.path.exists(malaria_file):
                print(f"Missing: {malaria_file}")
                continue
            if not os.path.exists(target_file):
                print(f"Missing: {target_file}")
                continue

            align_and_save(session, malaria_file, target_file, count, log_dir)

# ChimeraX will call this with its session object
main(session)


In [2]:
#run the code like this:
1.
#chimerax --nogui align_with_chimerax.py

2.
Or inside ChimeraX GUI via:
#Tools > Utilities > Run Python Script


In [ ]:
#this is with added rmsd calculation


def align_and_save(session, malaria_path, target_path, count, aligned_dir, log_dir):
    from chimerax.core.commands import run, run_on_session

    run(session, "log clear")
    run(session, f"open {target_path}")   # model #1
    run(session, f"open {malaria_path}")  # model #2
    run(session, "matchmaker #2 to #1")

    # Save aligned PDB
    aligned_filename = f"aligned_{count}_{os.path.basename(target_path).replace('.pdb', '')}_vs_{os.path.basename(malaria_path).replace('.pdb', '')}.pdb"
    aligned_path = os.path.join(aligned_dir, aligned_filename)
    log_path = os.path.join(log_dir, f"output{count}.txt")

    run(session, f"save {aligned_path}")
    run(session, f"log save {log_path}")

    # --- RMSD Calculation ---
    try:
        # Get model IDs for malaria and target
        malaria_model = session.models.list()[1]  # opened second
        target_model = session.models.list()[0]   # opened first

        # Find chain M in malaria and chain D in target
        chain_M = [c for c in malaria_model.chains if c.chain_id == 'M']
        chain_D = [c for c in target_model.chains if c.chain_id == 'D']

        if chain_M and chain_D:
            atoms_M = chain_M[0].atoms
            atoms_D = chain_D[0].atoms

            # Use only matching atoms by name for RMSD
            atom_pairs = [(a1, a2) for a1 in atoms_M for a2 in atoms_D if a1.name == a2.name]
            if not atom_pairs:
                print(f"No matching atoms for RMSD in pair {count}")
            else:
                from chimerax.atomic.struct_edit import rmsd
                rmsd_value = rmsd([a1 for a1, _ in atom_pairs], [a2 for _, a2 in atom_pairs])
                print(f"RMSD (chain M vs D) for pair {count}: {rmsd_value:.3f}")
        else:
            print(f"Chain M or D not found in pair {count}")
    except Exception as e:
        print(f"RMSD calculation failed for pair {count}: {e}")

    run(session, "close all")
